# Test explicit weight decay with the three-epoch patch model

Question: does AdamW weight decay improve development generalization without losing stage-detection scores?

Reuse the measured seed-42 three-epoch patch reference on all five original folds.
Train only five new candidate fits with decay 0.01. Exclude biases and LayerNorm parameters from decay.
Keep the fifteen-minute raw context, six features, width 16, dropout 0.2, rate 0.001, and budget 40/12 fixed.
Both models have 2,756 parameters and ten temporal tokens at full history.

Verify zero-decay AdamW against the current Adam path before preparation, preflight, or training.
This control uses synthetic inputs only. A matching short control does not prove identical full participant-training trajectories.

Primary outcome: paired mean fold macro-F1. Inspect NLL, stage trade-offs, participants, and temporal slices as secondary evidence.
Lower NLL alone does not establish better stage classification.
Keep the three final-test participants reserved.

Read docs/patch-adamw.md before enabling preparation or training.


## Locate the checkout and declare execution switches

**What:** Find the repository, configure logging, and declare optional setup, preparation, checks, and training.

**Why:** Existing local and cloned checkouts need the same paths, while model fitting requires explicit opt-in.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("patch_adamw_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-patch-adamw-grouped.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/patch-adamw-decay"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = True
RUN_TRAINING = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s preparation=%s preflight=%s training=%s",
            RUN_SETUP, RUN_PREPARATION, RUN_CHECK, RUN_TRAINING)


## Optional environment setup

**What:** Optionally clone this working branch and install Astral uv plus locked dependency groups.

**Why:** A fresh machine needs project code and the same dependencies. The existing local environment can skip this cell.
Fresh clones also require development features and the completed five-fold reference artifacts.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the full-fold reference and fixed controls

**What:** Verify reference artifacts, scores, participant assignments, device, architecture, and every unchanged training setting.

**Why:** A weight-decay comparison needs the same inputs and evaluation targets. Reused measurements must match the current recipe.
The only recipe changes are the declared optimizer and artifact output directory.


In [ ]:
import json

import pandas as pd
import yaml

from sleep_classifier.experiment_evidence import EpochEvidenceReport
from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.neural_training import TrainingConfig
from sleep_classifier.paired_architecture_report import PairedArchitectureReport

experiment = GroupedValidationExperiment(ROOT / "configs/transformer-patch-adamw-grouped.yaml")
reference = GroupedValidationExperiment(
    ROOT / "configs/transformer-patch-grouped.yaml"
)
plan = experiment.plan()

reference_summary = reference.output_dir / "summary.json"
if not reference_summary.exists():
    raise FileNotFoundError("The completed five-fold three-epoch patch reference is required")

reference_saved = json.loads(reference_summary.read_text())
if reference_saved["plan"] != reference.plan():
    raise ValueError("The saved reference differs from its current participant-fold plan")

reference_validation = PairedArchitectureReport(reference_summary, reference_summary).build()
reference_manifest = json.loads((reference.output_dir / "manifest.json").read_text())
resolved_device = str(TrainingConfig(experiment.model_settings).device())
if resolved_device != reference_manifest["device"]:
    raise ValueError("Candidate device differs from the measured reference device")
if plan["folds"] != reference_saved["plan"]["folds"]:
    raise ValueError("Candidate participant assignments differ from the original five folds")
if len(plan["folds"]) != 5 or plan["maximum_epochs"] != 200:
    raise ValueError("The candidate must use five new fits with a 200-epoch maximum")
if len(plan["development"]) != 26 or len(plan["reserved_test"]) != 3:
    raise ValueError("Development or reserved-test membership differs from the frozen cohort")

excluded = {"optimizer", "output_dir"}
recipe = {key: value for key, value in experiment.model_settings.items() if key not in excluded}
reference_recipe = {key: value for key, value in reference.model_settings.items()
                    if key not in excluded}
if recipe != reference_recipe:
    raise ValueError("Another optimization, feature-context, or Transformer-core setting changed")
if (experiment.model_settings["model_type"] != "patch_transformer"
        or experiment.model_settings["patch_size"] != 3
        or reference.model_settings["patch_size"] != 3
        or experiment.model_settings["learning_rate"] != 0.001):
    raise ValueError("The candidate must retain three-epoch patches and rate 0.001")
expected_optimizer = {"name": "adamw", "weight_decay": 0.01, "exclude_bias_and_norm": True}
if experiment.model_settings.get("optimizer") != expected_optimizer:
    raise ValueError("The candidate must use the predefined AdamW decay policy")
if reference.model_settings.get("optimizer") is not None:
    raise ValueError("The reused reference must retain its original implicit Adam recipe")

# Match the measured fold recipes as well as the current YAML
artifact_fields = {"prepare_config", "output_dir", "device"}
expected_measured_recipe = {
    key: value for key, value in reference.model_settings.items() if key not in artifact_fields
}
for result in reference_saved["results"]:
    metrics_path = Path(result["metrics_path"])
    if not metrics_path.is_absolute():
        metrics_path = reference_summary.parent / metrics_path
    measured_config = json.loads(metrics_path.read_text())["config"]
    measured_recipe = {key: value for key, value in measured_config.items()
                       if key not in artifact_fields}
    if measured_recipe != expected_measured_recipe:
        raise ValueError("The current reference YAML differs from its measured fold recipe")
diagnostic_controls = yaml.safe_load((ROOT / "configs/transformer-diagnostics.yaml").read_text())
if experiment.model_settings["diagnostics"] != diagnostic_controls["diagnostics"]:
    raise ValueError("Candidate evidence settings differ from the validated diagnostic controls")
if (experiment.model_settings.get("lr_scheduler")
        or experiment.model_settings.get("optimizer_agent")):
    raise ValueError("This weight-decay comparison requires a constant rate without an agent")
display(pd.DataFrame(plan["folds"]).set_index("name"))
display(pd.DataFrame([{"raw_context_epochs": 30, "patch_size_epochs": 3,
                       "maximum_patch_tokens": 10, "new_fits": 5,
                       "maximum_new_epochs": 200, "device": resolved_device}]))

logger.info("Reference verified: same five folds, raw history, features, and training controls")
logger.info("Only the optimizer decay recipe changes; diagnostic controls remain identical")

## Verify the zero-decay optimizer control first

**What:** Compare legacy Adam and grouped AdamW at zero decay on twenty synthetic updates, on CPU and the selected device.
Show parameter, gradient, loss, and optimizer-state agreement. Save the control report outside the candidate run directory.

**Why:** A hidden optimizer-path change could confound the decay experiment. This check reads no participant data.
The check restores random states before it returns. It does not consume training examples or fit preprocessing.
A short synthetic match supports reference reuse; it does not prove bitwise identity over every future training trajectory.
Stop if a control fails. Do not enable candidate training until every displayed control passes.


In [ ]:
from sleep_classifier.optimizer_control import ZeroDecayControl

control_devices = list(dict.fromkeys(["cpu", resolved_device]))
zero_decay_controls = [
    ZeroDecayControl(experiment.model_settings, device=device).run()
    for device in control_devices
]
if not all(result["passed"] for result in zero_decay_controls):
    raise RuntimeError("Zero-decay AdamW does not match the legacy Adam control")

display(pd.DataFrame([
    {key: value for key, value in result.items() if key != "groups"}
    for result in zero_decay_controls
]).set_index("device"))
display(pd.DataFrame(zero_decay_controls[-1]["groups"]))
control_path = ROOT / "artifacts/transformer-patch-adamw-zero-control.json"
control_path.parent.mkdir(parents=True, exist_ok=True)
control_path.write_text(json.dumps(zero_decay_controls, indent=2) + "\n")
logger.info("Zero-decay controls passed; saved %s", control_path)


## Prepare only the weight-decay candidate

**What:** Optionally prepare five candidate folds with frozen recipes, provenance, and copied unscaled features.

**Why:** Each fold needs training-only preprocessing and participant-disjoint validation. The completed reference must remain unchanged.
Keep RUN_PREPARATION false after the candidate manifest exists.


In [ ]:
if RUN_PREPARATION:
    if not all(result["passed"] for result in zero_decay_controls):
        raise RuntimeError("Zero-decay control must pass before preparation")
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference participant-fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
        --config configs/transformer-patch-adamw-grouped.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Preparation failed; inspect existing outputs before retrying")
else:
    logger.info("Patch candidate preparation skipped")

## Check every prepared fold and inspect the actual optimizer policy

**What:** Run a no-gradient forward check on all five candidate configurations and show each optimizer's decay groups.

**Why:** Validate real shapes, device selection, and parameter assignment before optimization.
The model remains unchanged. Every trainable parameter must occur once, with biases and LayerNorm parameters exempt.


In [ ]:
if RUN_CHECK and (experiment.output_dir / "manifest.json").exists():
    if not all(result["passed"] for result in zero_decay_controls):
        raise RuntimeError("Zero-decay control must pass before preflight")
    %cd "{ROOT}"
    for fold in plan["folds"]:
        fold_config = experiment.output_dir / fold["name"] / "gru.yaml"
        logger.info("Checking %s without optimization", fold["name"])
        !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
            --config "{fold_config}" --check
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Patch candidate preflight failed")
elif RUN_CHECK:
    logger.info("Preflight pending: prepare the five folds first")
else:
    logger.info("Preflight skipped")

## Train the five weight-decay models when ready

**What:** Repeat the synthetic control on the selected device, then optionally train five fresh candidate fits.

**Why:** Rechecking at launch prevents an earlier notebook result from authorizing an unverified optimizer path.
This cell never retrains the reference. It requires RUN_TRAINING = True and a valid prepared manifest.
Maximum new work is five fits and 200 epochs. Existing results block accidental reruns.


In [ ]:
if RUN_TRAINING:
    launch_control = ZeroDecayControl(experiment.model_settings, device=resolved_device).run()
    if not launch_control["passed"]:
        raise RuntimeError("Zero-decay control failed at training launch")
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference participant-fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.grouped_validation \
        --config configs/transformer-patch-adamw-grouped.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Patch training failed; inspect retained evidence before retrying")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")

## Inspect completion status and the measured reference

**What:** Show available reference/candidate fold status and the reference's pooled selected-checkpoint scores.

**Why:** A complete reference does not imply a complete candidate. Pending runs must not become fabricated performance scores.


In [ ]:
reporters = {"reference": EpochEvidenceReport(reference.output_dir),
             "candidate": EpochEvidenceReport(experiment.output_dir)}
evidence_by_role = {role: reporter.build() for role, reporter in reporters.items()}
status_rows = [{"recipe": role, "study_status": evidence["status"], "fold": fold["fold"],
                "fold_status": fold["status"], "recorded_epochs": len(fold["epochs"])}
               for role, evidence in evidence_by_role.items() for fold in evidence["folds"]]
if status_rows:
    display(pd.DataFrame(status_rows).set_index(["recipe", "fold"]))
else:
    logger.info("The candidate has not been prepared")
display(pd.DataFrame([{"recipe": "measured reference",
                       **reference_validation["reference"]}]).drop(
    columns=["confusion_matrix", "classes"],
))

## Inspect available learning curves

**What:** Plot existing epoch losses, macro-F1, gradients, and actual learning rates for both recipes.

**Why:** Both recipes save passive diagnostics. Compare overfitting and confidence errors without replacing the primary score.

In [ ]:
import matplotlib.pyplot as plt

for role, reporter in reporters.items():
    if not any(fold["epochs"] for fold in evidence_by_role[role]["folds"]):
        logger.info("%s has no saved per-epoch evidence; no curve is reconstructed", role)
        continue
    figure = reporter.plot_curves()
    figure.suptitle(f"{role}: completed-epoch evidence")
    figure.tight_layout(rect=(0, 0, 1, 0.97))
    display(figure)
    plt.close(figure)

## Compare selected checkpoints only after all five fits finish

**What:** Verify matched target identities and training controls, then compare fold, pooled, and class scores.

**Why:** The primary score is mean fold macro-F1. Pooled accuracy alone can hide REM or Deep losses.
All five validation folds also select checkpoints; this remains development evaluation, not the final reserved test.
Stage indices are 0 = Wake, 1 = Light, 2 = Deep, and 3 = REM.


In [ ]:
comparison = None
candidate_summary = experiment.output_dir / "summary.json"
if candidate_summary.exists() and json.loads(candidate_summary.read_text())["status"] == "complete":
    comparison = PairedArchitectureReport(
        reference_summary, candidate_summary, comparison="weight_decay",
    ).build()
    display(pd.DataFrame(comparison["folds"]).set_index("fold"))
    score_rows = [{"recipe": role, "accuracy": comparison[role]["accuracy"],
                   "pooled_macro_f1": comparison[role]["macro_f1"]}
                  for role in ("reference", "candidate")]
    display(pd.DataFrame(score_rows).set_index("recipe"))
    class_rows = [{"recipe": role, "stage": stage, **scores}
                  for role in ("reference", "candidate")
                  for stage, scores in comparison[role]["classes"].items()]
    display(pd.DataFrame(class_rows).set_index(["recipe", "stage"]))
    destination = experiment.output_dir / "paired-comparison.json"
    destination.write_text(json.dumps(comparison, indent=2) + "\n")
    logger.info("Mean paired fold macro-F1 difference: %.6f",
                comparison["paired"]["mean_macro_f1_difference"])
else:
    logger.info("The patch candidate is incomplete; no paired conclusion is available")

## Visualize the pooled confusion matrices

**What:** Plot selected-checkpoint counts for true and predicted stages in the same order for both recipes.

**Why:** The matrices expose systematic stage substitutions that an aggregate score can hide.
Rows are true stages and columns are predicted stages. Raw counts reflect the cohort's class imbalance.


In [ ]:
if comparison is not None:
    import numpy as np
    from sklearn.metrics import ConfusionMatrixDisplay

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    for axis, role in zip(axes, ("reference", "candidate"), strict=True):
        ConfusionMatrixDisplay(
            np.asarray(comparison[role]["confusion_matrix"]),
            display_labels=["Wake", "Light", "Deep", "REM"],
        ).plot(ax=axis, colorbar=False, values_format="d")
        axis.set_title(role)
    figure.tight_layout()
    display(figure)
    plt.close(figure)
else:
    logger.info("Paired confusion matrices are pending candidate completion")

## Inspect candidate selected-checkpoint probability loss

**What:** Read selected-epoch validation NLL and evaluation-mode training NLL, then calculate pooled validation NLL.

**Why:** Negative log-likelihood measures probability assigned to the true stage; lower is better.
A confident wrong prediction receives a large penalty. NLL complements hard-label F1 and does not select our checkpoints.
Weight each fold by target count. Keep partial evidence separate from a complete pooled score.

In [ ]:
candidate_rows = []
selected_loss_sum = 0.0
selected_target_count = 0
for fold in evidence_by_role["candidate"]["folds"]:
    selected = next((epoch for epoch in fold["epochs"]
                     if epoch["epoch"] == fold["best_epoch"]), None)
    if selected is None:
        continue
    count = sum(stage["count"] for stage in selected["validation_classes"].values())
    selected_target_count += count
    selected_loss_sum += selected["validation_nll"] * count
    candidate_rows.append({"fold": fold["fold"], "selected_epoch": fold["best_epoch"],
                           "selected_validation_nll": selected["validation_nll"],
                           "selected_training_evaluation_nll": selected["training_evaluation_nll"],
                           "scored_targets": count})
if candidate_rows:
    display(pd.DataFrame(candidate_rows).set_index("fold"))
    if len(candidate_rows) == 5:
        logger.info("Candidate pooled selected-checkpoint NLL: %.6f",
                    selected_loss_sum / selected_target_count)
    else:
        logger.info("Partial fold NLL only; no complete candidate aggregate yet")
else:
    logger.info("Selected-checkpoint NLL is pending candidate completion")

## Compare short histories and stage transitions

**What:** Compare matched selected-checkpoint predictions by available history and at true stage transitions.
Show accuracy, macro-F1, class precision, recall, F1, and support for both recipes.

**Why:** A global score can hide startup or stage-specific regressions from regularization.
History bins are disjoint; the transition slice overlaps them. Histories restart at participant and segment boundaries.
Ground-truth preceding stages define evaluation slices only. They never enter the model.
Small supports limit conclusions. Treat these development slices as exploratory, and keep checkpoint selection unchanged.


In [ ]:
from sleep_classifier.temporal_slice_report import TemporalSliceReport

if comparison is not None:
    temporal_report = TemporalSliceReport.from_grouped(
        reference_summary, candidate_summary, comparison="weight_decay",
    ).build()
    slice_rows = []
    slice_class_rows = []
    for row in temporal_report["slices"]:
        for role in ("reference", "candidate"):
            scores = row[role]
            if scores is None:
                continue
            slice_rows.append({"slice": row["name"], "recipe": role,
                               "targets": row["targets"], "accuracy": scores["accuracy"],
                               "macro_f1": scores["macro_f1"]})
            slice_class_rows.extend(
                {"slice": row["name"], "recipe": role, "stage": stage, **values}
                for stage, values in scores["classes"].items()
            )
    display(pd.DataFrame(slice_rows).set_index(["slice", "recipe"]))
    display(pd.DataFrame(slice_class_rows).set_index(["slice", "recipe", "stage"]))
    (experiment.output_dir / "temporal-slices.json").write_text(
        json.dumps(temporal_report, indent=2) + "\n",
    )
else:
    logger.info("Temporal slices require five complete paired candidate fits")

## Save the local evidence summary

**What:** Regenerate the candidate's portable local evidence report without loading or changing its model.

**Why:** The same inspector supports prepared, partial, failed, and completed studies.
An unprepared report does not create a directory that would block later preparation.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.experiment_evidence \
    --output-dir artifacts/transformer-patch-adamw-grouped-validation --report
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Patch evidence inspection failed")

## Export the executed HTML report

**What:** Execute report and preflight cells through a fresh kernel, then save local HTML.

**Why:** The source notebook remains output-free, while a reproducible executed report preserves inspection results.
Automated export skips setup, preparation, training, and this recursive export cell regardless of execution switches.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/26_patch_adamw.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Patch notebook export failed")